In [0]:
%sql
create volume if not exists spark_lab

In [0]:
import requests
#fetch unity catalog name
catalog_name=spark.sql("select current_catalog()").collect()[0][0]
print(catalog_name)
volume_base=f"/Volumes/{catalog_name}/default/spark_lab"

files=["2019.csv","2020.csv","2021.csv"]

#download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/refs/heads/main/DataAnalytics/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

In [0]:
df=spark.read.format('csv').load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv')


In [0]:
display(df.limit(5))

In [0]:
#define schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, FloatType

schema=StructType([
    StructField('SalesOrderNumber',StringType()),
    StructField("SalesorderLineNumber",IntegerType()),
    StructField("OrderDate",DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email",StringType()),
    StructField("Item",StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())

])

df=spark.read.format("csv").load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', schema=schema)

display(df.limit(10))

In [0]:
df.createOrReplaceTempView("salesorders")
spark_df=spark.sql("select * from salesorders")
display(spark_df)

In [0]:
sql_query="select year(OrderDate) as Orderyear, sum((UnitPrice*Quantity)+Tax) as GrossRevenue from salesorders group by year(OrderDate) order by year(OrderDate) desc"

df_spark=spark.sql(sql_query)
df_spark.show()

In [0]:
spark_df.write.mode('overwrite').saveAsTable('dbws01.default.sales_orders')

In [0]:
import matplotlib.pyplot as plt

df_sales=df_spark.toPandas()


In [0]:
plt.bar(x=df_sales['Orderyear'], height=df_sales['GrossRevenue'])
plt.show()

In [0]:
import seaborn as sns
plt.clf()

sns.barplot(x=df_sales['Orderyear'], y=df_sales['GrossRevenue'], data=df_sales)
plt.show()

In [0]:
%pip install langchain langchain-community langchain-openai


In [0]:
dbutils.library.restartPython()

In [0]:
import langchain